In [ ]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from gold_common_functions import connect_to_postgres
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
SOURCE_QUERY = """
SELECT
	p.*,
	c.category_final
FROM gold.products AS p
JOIN gold.categories AS c
	ON p.category_key = c.category_key 
WHERE 
	p.snapshot_date = '2025-12-07'
"""

In [ ]:
conn = connect_to_postgres()
if conn:
    df = pd.read_sql(SOURCE_QUERY, conn)
    conn.close()

In [ ]:
df['final_price'] = df['final_price'].astype(float)

In [ ]:
df.info()

# Categories

In [ ]:
# 1. Agregación de todas las métricas necesarias (count, max, min, mean, median)
# Filtramos (no 'Otros')
df_metrics = df[df['category_final'] != 'Otros'].groupby(['category_final', 'supermarket'])['final_price'].agg(
    count='count',
    max_price='max',
    min_price='min',
    mean_price='mean',
    median_price='median'
).reset_index()

category_final_list = df_metrics['category_final'].unique()

In [ ]:
# Iteramos sobre cada categoria
for category in category_final_list:
    category_data = df_metrics[df_metrics['category_final'] == category]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.suptitle(f'Análisis de Precios Agregados - Categoria: {category}', fontsize=16, y=1.02)

    # --- PLOT 1: Count (Cantidad de Productos) ---
    sns.barplot(
        data=category_data, 
        x='supermarket', 
        y='count', 
        ax=axes[0],
        color='skyblue'
    )
    axes[0].set_title('Conteo de Productos')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=45)
    
    # --- PLOT 2: Max y Min Price (Range) ---
    range_data = category_data[['supermarket', 'max_price', 'min_price']].melt(
        id_vars='supermarket', 
        var_name='Metric', 
        value_name='Price'
    )
    sns.barplot(
        data=range_data, 
        x='supermarket', 
        y='Price', 
        hue='Metric', 
        ax=axes[1]
    )
    axes[1].set_title('Precio Máximo y Mínimo')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)

    # --- PLOT 3: Mean y Median Price ---
    central_data = category_data[['supermarket', 'mean_price', 'median_price']].melt(
        id_vars='supermarket', 
        var_name='Metric', 
        value_name='Price'
    )
    sns.barplot(
        data=central_data, 
        x='supermarket', 
        y='Price', 
        hue='Metric', 
        ax=axes[2]
    )
    axes[2].set_title('Media vs. Mediana')
    axes[2].set_xlabel('')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# TOM

### Preparacion

In [ ]:
df['search_term'] = (
    df['product_name']
    .str.lower()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

tom_brands = {
    'Pechugón': r'pechugon',
    'Cavallaro': r'cavallaro',
    'Tres Leones': r'tres\s*leones', # \s* maneja 0 o más espacios
    'Lactolanda': r'lactolanda',
    'Kurupí': r'kurupi',
    'Ochsi': r'ochsi',
    'Hellmann\'s': r'hellmann',
    'Trébol': r'trebol',
    'Coca-Cola': r'coca.*cola', # .* permite "Coca Cola" o "Coca-Cola"
    'Sedal': r'sedal',
    'Pedigree': r'pedigree',
    'Mar': r'\bmar\b', # \b asegura palabra exacta (evita "maravilla")
    'Nikito': r'nikito',
    'OMO': r'\bomo\b',
    'Guaraní': r'guarani',
    'Mapex': r'mapex',
    'Caricias': r'caricia',
    'Ades': r'\bades\b',
    'Anita': r'\banita\b',
    'Brahma': r'brahma',
    'Red Bull': r'red\s*bull',
    'Nescafé': r'nescafe',
    'Huggies': r'huggies',
    'Incabril': r'incabril',
    'Colgate': r'colgate',
    'Rexona': r'rexona',
    'Nestlé': r'nestle'
}

In [ ]:
df['tom_brand'] = 'Otros' # Valor por defecto

for brand_label, pattern in tom_brands.items():
    mask = df['search_term'].str.contains(pattern, regex=True, na=False)
    df.loc[mask, 'tom_brand'] = brand_label

df.drop(columns=['search_term'], inplace=True)

print(df[df['tom_brand'] != 'Otros']['tom_brand'].value_counts())

In [ ]:
df['product_name'][df["tom_brand"] == "Huggies"].sample(10)

In [ ]:
df_chart = df[['supermarket', 'final_price']][df["tom_brand"] == "Lactolanda"].copy()
df_chart = df_chart.groupby('supermarket')['final_price'].agg(["count","sum"]).reset_index()
df_chart.head()

In [ ]:
datos = df[df["tom_brand"] == "Lactolanda"].groupby('supermarket')['final_price'].agg(["count", "sum"])
datos.plot(kind='bar', secondary_y='sum', figsize=(10, 5), title="Lactolanda: Cantidad vs. Precio Total")

### Visualizacion

In [ ]:
# 1. Agregación de todas las métricas necesarias (count, max, min, mean, median)
# Filtramos (no 'Otros')
df_metrics = df[df['tom_brand'] != 'Otros'].groupby(['tom_brand', 'supermarket'])['final_price'].agg(
    count='count',
    max_price='max',
    min_price='min',
    mean_price='mean',
    median_price='median'
).reset_index()

tom_brands_list = df_metrics['tom_brand'].unique()

In [ ]:
# Iteramos sobre cada marca TOM
for brand in tom_brands_list:
    brand_data = df_metrics[df_metrics['tom_brand'] == brand]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.suptitle(f'Análisis de Precios Agregados - Marca: {brand}', fontsize=16, y=1.02)

    # --- PLOT 1: Count (Cantidad de Productos) ---
    sns.barplot(
        data=brand_data, 
        x='supermarket', 
        y='count', 
        ax=axes[0],
        color='skyblue'
    )
    axes[0].set_title('Conteo de Productos')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=45)
    
    # --- PLOT 2: Max y Min Price (Range) ---
    range_data = brand_data[['supermarket', 'max_price', 'min_price']].melt(
        id_vars='supermarket', 
        var_name='Metric', 
        value_name='Price'
    )
    sns.barplot(
        data=range_data, 
        x='supermarket', 
        y='Price', 
        hue='Metric', 
        ax=axes[1]
    )
    axes[1].set_title('Precio Máximo y Mínimo')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)

    # --- PLOT 3: Mean y Median Price ---
    central_data = brand_data[['supermarket', 'mean_price', 'median_price']].melt(
        id_vars='supermarket', 
        var_name='Metric', 
        value_name='Price'
    )
    sns.barplot(
        data=central_data, 
        x='supermarket', 
        y='Price', 
        hue='Metric', 
        ax=axes[2]
    )
    axes[2].set_title('Media vs. Mediana')
    axes[2].set_xlabel('')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()